# PyTorch's example to demonstrate Amazon SageMaker Heterogeneous Cluster for model training


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

---

### Description
Heterogeneous clusters enable launching training jobs that use multiple instance types in a single job. This capability can improve your training cost and speed by running different parts of the model training on the most suitable instance type. This use case typically happens in computer vision DL training, where training is bottleneck on CPU resources needed for data augmentation, leaving the expensive GPU underutilized. Heterogeneous clusters enable you to add more CPU resources to fully utilize GPUs, thus increase training speed and cost-efficiency. For more details, you can find the documentation of this feature [here](https://docs.aws.amazon.com/sagemaker/latest/dg/train-heterogeneous-cluster.html).

This notebook demonstrates how to use Heterogeneous Cluster feature of SageMaker Training with PyTorch.  The notebook works on Python 3 (_PyTorch 1.12 Python 3.8 CPU Optimized_) image of SageMaker Studio Notebook instance, and runs on _ml.t3.medium_ instance type.

The notebook covers:
- Setting up SageMaker Studio Notebook 
- Setting up the Training environment 
- Submit a Training job
- Monitor and visualize the CloudWatch metrics
- Comparing time-to-train and cost-to-train
- Considerations
- Conclusion 

In this sample notebook, we have taken the PyTorch model based on this [official MNIST example](https://github.com/pytorch/examples/tree/main/MNIST). We modified the training code to be heavy on data pre-processing. We are going to train this model in both Homogeneous and Heterogeneous Cluster modes. The flag to train on any of these modes can be set using `IS_HETERO = False or True` in section **B.2 Configure environment variables**. 

Homogeneous Training Job - In this baseline we observe an ml.p3.2xlarge with an under-utilized GPU due to a CPU bottleneck. 
<img src=images/homogeneous-cluster-diagram.png alt="homogeneous-training job" />  

Heterogeneous Training Job - Where we add ml.c5.9xlarge instance for extra CPU cores, to allow increased GPU usage of ml.p3.2xlarge instance, and improve cost-efficiency. Both the jobs runs the training code, train data set, pre-processing, and other relevant parameters.
<img src=images/heterogeneous-cluster-diagram.png alt="heterogeneous-training job" />

In homogeneous cluster training job, the data pre-processing and Deep Neural Network (DNN) training code runs on the same instance. However, in heterogeneous cluster training job, the data pre-processing code runs on the CPU nodes (here by referred as **data_group or data group**), whereas the Deep Neural Network (DNN) training code runs on the GPU nodes (here referred as **dnn_group or dnn group**). The inter-node communication between the data and dnn groups is handled by generic implementation of [gRPC client-server interface](https://grpc.io/docs/languages/python/basics/).  

The script (`launcher.py`) has the logic to detect (using SageMaker environment variables) whether the node it is running on belongs to data_group or dnn_group. If it is data_group, it spawns a separate process by executing `train_data.py`. This script runs grpc-server service for extracting processed training batches using [Protocol Buffers](https://developers.google.com/protocol-buffers/docs/overview). The gRPC server running on the data_group listens on a specific port (ex. 6000). In the code (`train_data.py`) documentation, we have chosen an implementation that keeps the data loading logic intact  where data batches are entered into a shared queue. The `get_samples` function of the `DataFeedService` pulls batches from the same queue and sends them to the client in the form of a continuous data stream. While fetching the data, the main entrypoint script `launcher.py` listens on port 16000 for a shutdown request coming from gRPC client i.e. data group. The `train_data.py` waits for shutdown action from the parent process. 

If the node belongs to dnn_group, the main training script (`launcher.py`) spawns a separate set of processes by executing `train_dnn.py`. The script runs gRPC client code and DNN component of the training job. It consumes the processed training data from the gRPC server. We have defined an iterable PyTorch dataset, RemoteDataset, that opens a connection to the gRPC server, and reads from a stream of data batches. Once the model is trained with all the batches of training data, the gRPC client exits, and the parent process`launcher.py` sends a shutdown request on port 16000. This indicates the gRPC server to shutdown, and signals ends of the training job. 

Here is how the workflow looks like:

<img src=images/pytorch-heterogeneous-workflow.png width=600px>

This example notebook runs a training job on 2 instances, 1 in each node group. The data_group uses ml.c5.9xlarge whereas dnn_group uses ml.p3.2xlarge.

This notebook refers following files and folders:

- Folders: 
  - `code`: this has the training (data pre-processing and dnn) scripts, and grpc client-server start and shutdown scripts
  - `images`: contains images referred in notebook
- Files: 
  - `launcher.py`: entry point training script. This script is executed on all the nodes irrespective of which group it belongs to. This is a parent process that makes a decision on where to spawn a data pre-processing or dnn component of the training job. The script runs on all the nodes as entry point. It also handles the shutdown logic for gRPC server. 
  - `train_data.py`, `dataset_feed_pb2.py`, `dataset_feed_pb2_grpc.py`: these scripts run on the data_group nodes and responsible for setting up grpc-server, start and shutdown.
  - `train_dnn.py`: this script runs dnn code on the training data set. It fetches preprocessed data from the data_group node as a stream using gRPC client-server communication. It also sends a shutdown request after all the iterations on the preprocessed training data set. 
  - `requirement.txt`: defines package required for gRPC 
  - `train.py`: this script is the entry point script for SageMaker homogeneous cluster training. This script is picked up when you choose IS_HETERO = False. This uses a local dataset and runs both data pre-processing and a dnn component on the same node. 

### Security groups update if running in private VPC
This section is relevant if you plan to [run in a private VPC](https://docs.aws.amazon.com/sagemaker/latest/dg/train-vpc.html) (passing `subnets` and `security_group_ids` parameters when defining an Estimator).  
SageMaker documentation recommends you [add](https://docs.aws.amazon.com/sagemaker/latest/dg/train-vpc.html#train-vpc-vpc) a rule for your security group that allows inbound connections between members of the same security group, for all TCP communication. This will also cover for the gRPC related traffic between instances:
- the data_group instances will listen on port 6000 for connections from all nodes. This stream is not encrypted. You can change the code to encrypted the connection if needed.
- the data_group intances listen on port 16000 for a shutdown signal from all nodes.

### A. Setting up SageMaker Studio notebook

#### Step 1 - Upgrade SageMaker SDK and dependent packages 
Heterogeneous Clusters for Amazon SageMaker model training was [announced](https://aws.amazon.com/about-aws/whats-new/2022/07/announcing-heterogeneous-clusters-amazon-sagemaker-model-training) on 07/08/2022. As a first step, ensure you have updated SageMaker SDK, PyTorch, and Boto3 client that enables this feature.

In [1]:
# [papermill-run] pip install disabled; V3 SDK preinstalled in execution env
# %%bash
# python3 -m pip install --upgrade boto3 botocore awscli sagemaker

#### Step 2 - Restart the notebook kernel 

In [2]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

#### Step 3 - Validate SageMaker Python SDK and PyTorch versions
Ensure the output of the cell below reflects:

- SageMaker Python SDK version 3.0 or above,
- boto3 1.24 or above 
- botocore 1.27 or above 
- PyTorch 1.10 or above 

In [3]:
# [papermill-run] version check disabled
# !pip show sagemaker torch boto3 botocore |egrep 'Name|Version|---'

--------------
### B. Setting up the Training environment

#### Step 1 - Import SageMaker components and set up the IAM role and Amazon S3 bucket

In [4]:
import os
import json
import datetime

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData, StoppingCondition, InstanceGroup
from sagemaker.core import image_uris

import boto3 as _boto3
sess = Session(boto_session=_boto3.Session(region_name="us-west-1"))  # [papermill-run] region-bound session

role = "arn:aws:iam::729646638167:role/SageMakerRole"  # [papermill-run] explicit role
region = sess.boto_region_name

default_bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix
default_bucket_prefix_path = ""

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    default_bucket_prefix_path = f"/{default_bucket_prefix}"

output_path = "s3://" + default_bucket + default_bucket_prefix_path + "/DEMO-MNIST"

print(role)
print(output_path)

[07/09/26 13:38:36] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=11087409;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=11087410;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=11087415;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=11087416;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

arn:aws:iam::729646638167:role/SageMakerRole
s3://sagemaker-us-west-1-729646638167/DEMO-MNIST


#### Step 2 - Configure environment variables 
This step defines whether you want to run training job in heterogeneous cluster mode or not. Also, defines instance groups, multiple nodes in group, and hyperparameter values. For baselining, run a homogeneous cluster training job by setting `IS_HETERO = False`. This will let both the data pre-processing and DNN code run on the same node i.e. `ml.p3.2xlarge`. 


Test configuration (if running training on p3.2xl or g5.2xl as dnn_group instance type, and c5.2xl as data_group instance type: (training duration: 7-8 mins)  
`num-data-workers: 4`  
`grpc-workers: 4`   
`num-dnn-workers: 4`  
`pin-memory": True`   
`iterations : 100`   

Performance configuration (if running training on p3.2xl as dnn_group instance type, and c5.9xl as data_group instance type OR training in homogeneous cluster mode i.e. g5.8xl): (training duration - 30 mins)  
`num-data-workers: 32`  
`grpc-workers: 2`   
`num-dnn-workers: 2`  
`pin-memory": True`   
`iterations : 4800`

Performance configuration (if running training on p3.2xl in homogeneous cluster mode):   
`num-data-workers: 8`  
`grpc-workers: 2`   
`num-dnn-workers: 2`  
`pin-memory": True`   
`iterations : 2400`

Note: This PyTorch example has not been tested with multiple instances in an instance group. 

In [5]:
IS_CLOUD_JOB = True
IS_HETERO = True  # if set to false, uses homogeneous cluster
PT_DATA_MODE = "service" if IS_HETERO else "local"  # local | service
IS_DNN_DISTRIBUTION = False  # Distributed Training with DNN nodes not tested, set it to False

# In V3, heterogeneous clusters are configured via Compute(instance_groups=[...])
# using sagemaker.train.configs.InstanceGroup. Note the V3 InstanceGroup takes
# keyword arguments (instance_group_name / instance_type / instance_count),
# unlike the V2 positional signature.
data_group = InstanceGroup(
    instance_group_name="data_group", instance_type="ml.c5.9xlarge", instance_count=1
)  # 36 vCPU #change the instance type if IS_HETERO=True
dnn_group = InstanceGroup(
    instance_group_name="dnn_group", instance_type="ml.g4dn.2xlarge", instance_count=1  # [papermill-run] us-west-1 has no P3; g4dn used for e2e only
)  # 8 vCPU #change the instance type if IS_HETERO=True

# DNN group instance type, used to select the training image and (optionally) distribution.
dnn_instance_type = "ml.g4dn.2xlarge"  # [papermill-run] us-west-1 has no P3; g4dn used for e2e only

hyperparameters = {
    "batch-size": 8192,
    "num-data-workers": 4,  # This number drives the avg. step time. More workers help parallel pre-processing of data. Recommendation: Total no. of cpu 'n' = 'num-data-wokers'+'grpc-workers'+ 2 (reserved)
    "grpc-workers": 4,  # No. of workers serving pre-processed data to DNN group (gRPC client). see above formula.
    "num-dnn-workers": 4,  # Modify this no. to be less than the cpu core of your training instances in dnn group
    "pin-memory": True,  # Pin to GPU memory
    "iterations": 100,  # No. of iterations in an epoch (must be multiple of 10).
    "region": sess.boto_region_name,
}

if IS_HETERO:
    compute = Compute(instance_groups=[data_group, dnn_group], volume_size_in_gb=10)
    entry_point = "launcher.py"
else:
    compute = Compute(
        instance_type="ml.p3.2xlarge",  # change the instance type if IS_HETERO=False
        instance_count=1,
        volume_size_in_gb=10,
    )
    entry_point = "train.py"

# NOTE (V2 -> V3): The original notebook contained an optional, untested
# `IS_DNN_DISTRIBUTION` branch that enabled MPI distributed training limited to
# the DNN instance group via a free-form `distribution` dict:
#
#     kwargs["distribution"] = {
#         "mpi": {
#             "enabled": True,
#             "processes_per_host": processes_per_host_dict[dnn_instance_type],
#             "custom_mpi_options": "--NCCL_DEBUG INFO",
#         },
#     }
#     if IS_HETERO:
#         kwargs["distribution"]["instance_groups"] = [dnn_group]
#
# In V3, distributed training is configured with a typed config object, e.g.
# `from sagemaker.train.distributed import MPI` and
# `ModelTrainer(..., distributed=MPI(process_count_per_node=N,
#                                    mpi_additional_options=["-x", "NCCL_DEBUG=INFO"]))`.
# However, the V3 `MPI`/`Torchrun` configs do NOT support binding the distributed
# runner to a specific instance group (there is no `instance_groups` field), so the
# original semantics of "run MPI only on the dnn_group" cannot be expressed directly.
# Since this branch was disabled by default and marked "not tested" upstream, it is
# left here as a reference only and is intentionally not activated.
if IS_DNN_DISTRIBUTION:
    raise NotImplementedError(
        "Limiting MPI distribution to a single instance group is not supported by the "
        "V3 distributed config. See the note above for details."
    )

#### Step 3: Set up the ModelTrainer
In order to use SageMaker to train our algorithm, we'll create a `ModelTrainer` that defines how to use the container to train. This includes the configuration we need to invoke SageMaker training. In V3, the framework-specific `PyTorch` estimator is replaced by the generic `ModelTrainer` together with a training image resolved via `image_uris.retrieve`.

In [6]:
# Resolve the PyTorch training image (replaces the V2 framework estimator's
# framework_version / py_version arguments).
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="1.11.0",  # 1.10.0 or later
    py_version="py38",  # Python v3.8
    instance_type=dnn_instance_type,
    image_scope="training",
)

source_code = SourceCode(
    source_dir="code",
    entry_script=entry_point,
    requirements="requirements.txt",
)

model_trainer = ModelTrainer(
    training_image=training_image,
    sagemaker_session=sess,
    role=role,
    source_code=source_code,
    compute=compute,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=4800),
    hyperparameters=hyperparameters,
    base_job_name="pt-hetero-H-" + str(IS_HETERO)[0],
)

[07/09/26 13:38:38] INFO     Role 'arn:aws:iam::729646638167:role/SageMakerRole' validated ]8;id=11087423;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=11087424;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             for training. Using it.                                                               

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=11087431;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=11087432;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-1-729646638167/pt-hetero-H-T'                  
                             kms_key_id=None compression_type='GZIP'                                               

                    INFO     Training image URI:                                               ]8;id=11087439;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=11087440;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-1.amazonaws.com/pytorch-training:1.1                     
                             1.0-gpu-py38                                                                          

#### Step 4: Download the MNIST Data and Upload it to S3 bucket

This is an optional step for now. The training job downloads the data on its run directly from MNIST website to the data_group node (grpc server). 

In [7]:
import logging
import boto3
from botocore.exceptions import ClientError

# Download training and testing data from a public S3 bucket


def download_from_s3(data_dir="./data", train=True):
    """Download MNIST dataset and convert it to numpy array

    Args:
        data_dir (str): directory to save the data
        train (bool): download training set

    Returns:
        None
    """

    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    if train:
        images_file = "train-images-idx3-ubyte.gz"
        labels_file = "train-labels-idx1-ubyte.gz"
    else:
        images_file = "t10k-images-idx3-ubyte.gz"
        labels_file = "t10k-labels-idx1-ubyte.gz"

    # download objects
    s3 = boto3.client("s3")
    bucket = f"sagemaker-example-files-prod-{sess.boto_region_name}"
    for obj in [images_file, labels_file]:
        key = os.path.join("datasets/image/MNIST", obj)
        dest = os.path.join(data_dir, obj)
        if not os.path.exists(dest):
            s3.download_file(bucket, key, dest)
    return


download_from_s3("./data", True)
download_from_s3("./data", False)

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=11087445;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=11087446;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [8]:
# Upload to the default bucket

prefix = "DEMO-MNIST"
bucket = sess.default_bucket()
loc = sess.upload_data(path="./data", bucket=bucket, key_prefix=prefix)

channels = [
    InputData(channel_name="training", data_source=loc),
    InputData(channel_name="testing", data_source=loc),
]

## C. Submit the training job

The job runs for the predefined iterations. DNN instance group sends a shutdown request to data group after done with the training. You can see the following entries in the CloudWatch logs of dnn instance. A job with 4800 iterations finishes in 29 mins in a Heterogeneous cluster composed of 1x ml.c5.9xlarge as data node and 1x ml.p3.2xlarge as DNN node.

Log excerpt from algo-1 (DNN instance)
```
4780: avg step time: 0.19709917231025106
INFO:__main__:4780: avg step time: 0.19709917231025106
4790: avg step time: 0.19694106239373696
INFO:__main__:4790: avg step time: 0.19694106239373696
4800: avg step time: 0.196784295383125
Saving the model
INFO:__main__:4800: avg step time: 0.196784295383125
INFO:__main__:Saving the model
Training job completed!
INFO:__main__:Training job completed!
Process train_dnn.py closed with returncode=0
Shutting downdata service dispatcher via: [algo-2:16000]
shutdown request sent to algo-2:16000
2022-08-16 01:15:05,555 sagemaker-training-toolkit INFO     Waiting for the process to finish and give a return code.
2022-08-16 01:15:05,555 sagemaker-training-toolkit INFO     Done waiting for a return code. Received 0 from exiting process.
2022-08-16 01:15:05,556 sagemaker-training-toolkit INFO     Reporting training SUCCESS
```

In [9]:
# In V3 the training job name is derived from `base_job_name` (set on the
# ModelTrainer above); ModelTrainer appends a unique timestamp automatically.
model_trainer.train(input_data_config=channels)

[07/09/26 13:38:39] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=11087453;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=11087454;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


[07/09/26 13:38:43] INFO     Creating training_job resource.                                     ]8;id=11087461;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11087462;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31123\31123]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=11087469;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=11087470;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

...

                    INFO     pt-hetero-H-T-20260709133839/algo-2-1783629568:                     ]8;id=11089517;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11089518;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             Shutting down data service dispatcher via: [algo-1:16000]                             

                    INFO     pt-hetero-H-T-20260709133839/algo-2-1783629568:                     ]8;id=11089523;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11089524;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             Shutdown request sent to algo-1:16000                                                 

                    INFO     pt-hetero-H-T-20260709133839/algo-2-1783629568:                     ]8;id=11089529;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11089530;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     pt-hetero-H-T-20260709133839/algo-2-1783629568:                     ]8;id=11089535;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11089536;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             Training Container Execution Completed                                                

[07/09/26 13:46:05] INFO     Final Resource Status: Completed                                    ]8;id=11089542;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=11089543;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31475\31475]8;;\

## D. Monitoring Instance Metrics for GPU and CPU utilization

Click on **View instance metrics** from the **Training jobs** node in **Amazon SageMaker Console**. In the run above, all 30 vCPU of Data node (algo-1) is approx. 100% utilized, and the GPU utilization is at 100% at frequent intervals in the DNN node (algo-2). To rescale the CloudWatch Metrics to 100% on CPU utilization for algo-1 and algo-2, use CloudWatch "Add Math" feature and average it out by no. of cores on those instance types.

<img src=images/heterogeneous-instance-metrics.png width=900px>

## E. Comparing time-to-train and cost-to-train

Let's continue with the above example i.e. train a heavy data pre-processing (CPU intensive) model (MNIST) requiring only 1 GPU. We start with ml.p3.2xlarge (1xV100 GPU, 8x vCPU) in homogeneous cluster mode to get the baseline performance numbers. Due to the no. of CPU cores, we could not go beyond 8 data loader/workers for data pre-processing. The avg. step cost was `7.6 cents` and avg. step time is `1.19 seconds`. 

Our objective is to reduce the cost and speed up the model training time. The first choice here is to scale up the instance type in the same family. If we leverage the next instance type (4 GPU) in the P3 family, the GPUs would have gone underutilized. In this case, we needed more vCPU to GPU ratio. Assuming we haven't had any instance type in another instance family or the model is incompatible with the CPU/GPU architectures of other instance families, we are constrained to use ml.p3.2xlarge. The only way then to have more vCPUs to GPU ratio is to use SageMaker feature, Heterogeneous Cluster, which enables customers to offload data pre-processing logic to CPU only instance types example ml.c5. In the next test, we offloaded CPU intensive work i.e. data preprocessing to ml.c5.9xlarge (36 vCPU) and continued using ml.p3.2xlarge for DNN. The avg. step cost was `1.9 cents` and avg. step time is `0.18 seconds`. 

In summary, we reduced the training cost by 4.75 times, and the avg. step reduced by 6.5 times. This was possible because with higher cpu count, we could use 32 data loader workers (compared to 8 with p3.2xl) to preprocess the data, and kept GPU close to 100% utilized at frequent intervals. Note: These numbers are just taken as a sample, you have to do benchmarking with your own model and dataset to come up with the exact price-performance benefits. 

## F. Considerations
This PyTorch example implementation of gRPC client-server supports a single instance in each instance group. You may like to extend the logic to support multiple instances in the instance group. 

## G. Conclusion
In this notebook, we demonstrated how to leverage heterogeneous cluster feature of SageMaker Training to achieve better price performance. To get started you can copy this example project, and only change the `train_dnn.py` script.


## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-heterogeneous_clusters_for_model_training|sm-heterogeneous_clusters_for_model_training.ipynb)
